In [50]:
import requests
import pandas as pd
import numpy as np

In [51]:
ALPHA_VANTAGE_KEY = "VGYPQW3II8BLVK4I"
TOMORROW_API_KEY = "lgI22KhugBYh8JkmCk3BeAH9XHn71Qex"

In [52]:
#Alpha Vantage API to make stock data

In [53]:
symbol = "IBM"

url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={ALPHA_VANTAGE_KEY}"

response = requests.get(url)
data = response.json()

print(data.keys())
#this code is to check what i can do in this API for free and check if the API works correctly (it was in documentation) 

dict_keys(['Meta Data', 'Time Series (Daily)'])


In [54]:
ts = data["Time Series (Daily)"]
print(list(ts.items())[:4])
#to see what most recent data i have, and i put first 4 to make it short

[('2026-04-02', {'1. open': '243.0000', '2. high': '248.2103', '3. low': '241.4850', '4. close': '248.1600', '5. volume': '3350858'}), ('2026-04-01', {'1. open': '242.1200', '2. high': '246.2618', '3. low': '240.1400', '4. close': '243.1400', '5. volume': '4228653'}), ('2026-03-31', {'1. open': '240.2700', '2. high': '242.8500', '3. low': '236.3800', '4. close': '242.3900', '5. volume': '4750293'}), ('2026-03-30', {'1. open': '237.8000', '2. high': '240.2100', '3. low': '236.1310', '4. close': '237.2500', '5. volume': '3882602'})]


In [55]:
stock_df = pd.DataFrame.from_dict(ts, orient="index")
stock_df.head()
#creating the dataset, "head" to see only 1st 5 rows

,1. open,2. high,3. low,4. close,5. volume
2026-04-02,243.0000,248.2103,241.4850,248.1600,3350858
2026-04-01,242.1200,246.2618,240.1400,243.1400,4228653
2026-03-31,240.2700,242.8500,236.3800,242.3900,4750293
2026-03-30,237.8000,240.2100,236.1310,237.2500,3882602
2026-03-27,238.8500,239.4400,233.7500,236.3400,4853707


In [56]:
stock_df.index = pd.to_datetime(stock_df.index)
stock_df = stock_df.sort_index()
stock_df = stock_df.reset_index()
stock_df.head()
#to put date in its own column and sort data to oldest to newest, I also reseted the indexes

,index,1. open,2. high,3. low,4. close,5. volume
0,2025-11-07,309.6800,310.0000,302.6301,306.3800,5070773
1,2025-11-10,306.8200,309.9400,304.2300,309.1300,2975188
2,2025-11-11,309.0000,317.9100,308.4300,313.7200,4381913
3,2025-11-12,319.8900,324.9000,314.5324,314.9800,6042686
4,2025-11-13,312.2900,314.6000,303.6800,304.8600,5310150


In [57]:
stock_df.columns = ["date", "open", "high", "low", "close", "volume"]
stock_df.head()
#renaming the columns 

,date,open,high,low,close,volume
0,2025-11-07,309.6800,310.0000,302.6301,306.3800,5070773
1,2025-11-10,306.8200,309.9400,304.2300,309.1300,2975188
2,2025-11-11,309.0000,317.9100,308.4300,313.7200,4381913
3,2025-11-12,319.8900,324.9000,314.5324,314.9800,6042686
4,2025-11-13,312.2900,314.6000,303.6800,304.8600,5310150


In [58]:
stock_df["open"] = pd.to_numeric(stock_df["open"], errors="coerce")
stock_df["high"] = pd.to_numeric(stock_df["high"], errors="coerce")
stock_df["low"] = pd.to_numeric(stock_df["low"], errors="coerce")
stock_df["close"] = pd.to_numeric(stock_df["close"], errors="coerce")
stock_df["volume"] = pd.to_numeric(stock_df["volume"], errors="coerce")

print(stock_df.dtypes)
#to make data as numbers and not as text, errors do if there is any missing data it will give NaN

date      datetime64[ns]
open             float64
high             float64
low              float64
close            float64
volume             int64
dtype: object


In [59]:
stock_df["daily_return"] = stock_df["close"].pct_change()
stock_df.head()
# to calculate the daily return based on the closing price. We compute the % change between current day's close price and current day's close price
#The result stored in the new column "daily_return"

,date,open,high,low,close,volume,daily_return
0,2025-11-07,309.68,310.00,302.6301,306.38,5070773,NaN
1,2025-11-10,306.82,309.94,304.2300,309.13,2975188,0.008976
2,2025-11-11,309.00,317.91,308.4300,313.72,4381913,0.014848
3,2025-11-12,319.89,324.90,314.5324,314.98,6042686,0.004016
4,2025-11-13,312.29,314.60,303.6800,304.86,5310150,-0.032129


In [60]:
stock_df = stock_df[["date", "close", "daily_return", "volume"]]
stock_df.head()
#I delited columns which no longer needed

,date,close,daily_return,volume
0,2025-11-07,306.38,NaN,5070773
1,2025-11-10,309.13,0.008976,2975188
2,2025-11-11,313.72,0.014848,4381913
3,2025-11-12,314.98,0.004016,6042686
4,2025-11-13,304.86,-0.032129,5310150


In [61]:
stock_df = stock_df.dropna(subset=["daily_return"])
stock_df.head()
#removing the 1st row with NaN

,date,close,daily_return,volume
1,2025-11-10,309.13,0.008976,2975188
2,2025-11-11,313.72,0.014848,4381913
3,2025-11-12,314.98,0.004016,6042686
4,2025-11-13,304.86,-0.032129,5310150
5,2025-11-14,305.69,0.002723,3592455


In [62]:
stock_df.to_csv("stock_data.csv", index=False)
#save clean data to csv file 

In [63]:
#Open Meteo API  to make weather data 

In [64]:
# i changed my API to this one bacause it has history weather data
#this is open API, so there is no key 
url = "https://archive-api.open-meteo.com/v1/archive?latitude=40.7128&longitude=-74.0060&start_date=2025-10-01&end_date=2026-04-01&daily=temperature_2m_mean,relative_humidity_2m_mean,cloudcover_mean,windspeed_10m_max&timezone=America/New_York"

response = requests.get(url)
weather_data = response.json()

print(weather_data.keys())

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])


In [65]:
#creating dataframe with daily variables
daily = weather_data["daily"]

weather_df = pd.DataFrame(daily)

weather_df.head()

,time,temperature_2m_mean,relative_humidity_2m_mean,cloudcover_mean,windspeed_10m_max
0,2025-10-01,16.3,52,13,14.4
1,2025-10-02,14.2,60,34,8.7
2,2025-10-03,15.5,67,34,8.9
3,2025-10-04,19.5,64,19,9.4
4,2025-10-05,21.4,62,2,9.6


In [66]:
#renaming columns so they will match to previous data (Alpha vantage) 
weather_df.columns = [
    "date",
    "temperature",
    "humidity",
    "cloud_cover",
    "wind_speed"
]

weather_df.head()

,date,temperature,humidity,cloud_cover,wind_speed
0,2025-10-01,16.3,52,13,14.4
1,2025-10-02,14.2,60,34,8.7
2,2025-10-03,15.5,67,34,8.9
3,2025-10-04,19.5,64,19,9.4
4,2025-10-05,21.4,62,2,9.6


In [67]:
weather_df.to_csv("weather_data.csv", index=False)

In [68]:
#Merging 2 datasets

In [69]:
stock_df['date'] = pd.to_datetime(stock_df['date'])
weather_df['date'] = pd.to_datetime(weather_df['date'])

merged_df = pd.merge(stock_df, weather_df, on="date", how="inner")

merged_df.head()

,date,close,daily_return,volume,temperature,humidity,cloud_cover,wind_speed
0,2025-11-10,309.13,0.008976,2975188,10.1,84,72,17.3
1,2025-11-11,313.72,0.014848,4381913,2.4,58,66,28.2
2,2025-11-12,314.98,0.004016,6042686,5.5,60,83,15.6
3,2025-11-13,304.86,-0.032129,5310150,7.3,65,25,21.1
4,2025-11-14,305.69,0.002723,3592455,6.3,64,37,16.5


In [70]:
merged_df["good_weather"] = (
    (merged_df["temperature"].between(5, 25)) &
    (merged_df["cloud_cover"] < 60) &
    (merged_df["wind_speed"] < 20)
).astype(int)
#my criteria for good weather 
merged_df.head()

,date,close,daily_return,volume,temperature,humidity,cloud_cover,wind_speed,good_weather
0,2025-11-10,309.13,0.008976,2975188,10.1,84,72,17.3,0
1,2025-11-11,313.72,0.014848,4381913,2.4,58,66,28.2,0
2,2025-11-12,314.98,0.004016,6042686,5.5,60,83,15.6,0
3,2025-11-13,304.86,-0.032129,5310150,7.3,65,25,21.1,0
4,2025-11-14,305.69,0.002723,3592455,6.3,64,37,16.5,1


In [71]:
merged_df["good_weather"].value_counts() #to look how many days match my criteria

good_weather
0    91
1     7
Name: count, dtype: int64

In [72]:
merged_df.to_csv("final_dataset.csv", index=False)

Regression description 

Dependent variable:
- daily_return

Independent variables:
 - good_weather (main calculated varible)
 - temperature
 - humidity
 - cloud_cover
 - wind_speed
First regression: 
daily_return = β0 + β1 * good_weather + β2 * temperature + β3 * humidity + β4 * cloud_cover + β5 * wind_speed + ε

Main regression: 
daily_return = β0 + β1 * good_weather

The goal of regression is to test is the good wheather influence on stock market returns. 